# Union H3 Tile Polygons into GeoJSON Boundary

This notebook reads Resolution 8 H3 tile polygons from `seed_ledger_level_1.csv`, merges them into a single unified polygon boundary using `geopandas` and `shapely`, and exports the result as a GeoJSON file.

In [4]:
import json
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads
from shapely.geometry import mapping

# 1. Load CSV tile data
csv_path = 'seed_ledger_level_1.csv'
df = pd.read_csv(csv_path)
df = df[df['checked']]

print(f"Loaded {len(df)} H3 tile polygons from {csv_path}")

Loaded 431 H3 tile polygons from seed_ledger_level_1.csv


In [5]:
# 2. Convert WKT geometry strings to Shapely geometries
df['geometry'] = df['geometry'].apply(loads)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

# 3. Merge all tile polygons into a unified geometry
union_geom = gdf.union_all() if hasattr(gdf, 'union_all') else gdf.unary_union

print(f"Merged Geometry Type: {union_geom.geom_type}")

Merged Geometry Type: Polygon


In [6]:
# 4. Construct GeoJSON FeatureCollection
geojson_output = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {
                "name": "London H3 Seed Union",
                "tile_count": len(df),
                "h3_res": int(df['h3_res'].iloc[0]) if 'h3_res' in df.columns else 8
            },
            "geometry": mapping(union_geom)
        }
    ]
}

# 5. Save output GeoJSON
output_path = 'london_union_boundary.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(geojson_output, f, indent=2)

print(f"Successfully exported merged GeoJSON to {output_path}")

Successfully exported merged GeoJSON to london_union_boundary.json
